<a href="https://colab.research.google.com/github/dee1empire/-ITAI-1371-ML-Labs-/blob/main/Copy_of_Responsible_AI_DeloresBledsoe_ITAI_1371.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OncoGuide AI - Responsible AI - Responsible AI Group Project (Cancer Diagnosis Support)


OncoGuide AI is a hypothetical AI assistant for clinicians.
It analyzes patient data to flag individuals who may be at higher risk of cancer, supporting earlier investigation and follow-up (NOT replacing doctors).


      **ROLES**




*   AI Ethicist
*   Data Scientist
*   Product Manager
*   Consumer Advocate
*   Government Regulator


This Notebook:


1. Simulates patient data.  
2. Trains a simple model to predict " high-cancer risk".
3. Demonstrates basic fairness and transparency ideas.
4. Embeds role-based ethical analysis.
5. Ends with a knowledge check.







**IMPORTS & BASIC SETUP**

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

np.random.seed(42)

**SIMULATION DATA (DATA SCIENTIST FOCUS)**

In [ ]:
# Simulate a dataset of patients for cancer risk screening
n_patients = 600

# Feature (simplified, synthetic):
# - age: years
# - tumor_marker_level: numeric biomarker (0-100)
# - family_history: 0 = no, 1 = yes
# - smoking_index: 0-20 (higher = more smoking exposure)
# - lifestyle_score: 0-10 (higher = healthier lifestyle)
# - sex: 0 = female, 1 = male (for fairness checks)

age = np.random.normal(loc=55, scale=10, size=n_patients).clip(20, 90)
tumor_marker_level = np.random.choice([0, 1], size=n_patients, p=[0.6, 0.4])
family_history = np.random.choice([0, 1], size=n_patients, p=[0.6, 0.4])
smoking_index = np.random.normal(loc=5, scale=4, size=n_patients).clip(0, 20)
lifestyle_score = np.random.normal(loc=6, scale=2, size=n_patients).clip(0, 10)
sex = np.random.choice([0, 1], size=n_patients, p=[0.5, 0.5])

# Target: high_risk (1 = flagged for further cancer investigation, 0 = not flagged)
risk_score = (
    (age - 50) * 0.03 +
    (tumor_marker_level - 40) * 0.04 +
    family_history * 0.5 +
    smoking_index * 0.06 -
    lifestyle_score * 0.08
)

prob_high_risk = 1 / (1 + np.exp(-risk_score))
high_risk = (prob_high_risk > np.median(prob_high_risk)).astype(int)

# Create a DataFrame
data = pd.DataFrame({
    'age': age,
    'tumor_marker_level': tumor_marker_level,
    'family_history': family_history,
    'smoking_index': smoking_index,
    'lifestyle_score': lifestyle_score,
    'sex': sex,
    'high_risk': high_risk
})

print('Sample of simulated patient data:')
print(data.head())

Sample of simulated patient data:
         age  tumor_marker_level  ...  sex  high_risk
0  59.967142                   0  ...    0          0
1  53.617357                   1  ...    1          0
2  61.476885                   1  ...    1          1
3  70.230299                   1  ...    0          0
4  52.658466                   1  ...    0          0

[5 rows x 7 columns]


In [ ]:
print("Unique values in column:", data['high_risk'].unique())
print("Value counts:", data['high_risk'].value_counts)

Unique values in column: [0 1]
Value counts: <bound method IndexOpsMixin.value_counts of 0      0
1      0
2      1
3      0
4      0
      ..
595    1
596    0
597    0
598    0
599    1
Name: high_risk, Length: 600, dtype: int64>


**TRAINING OF SIMPLE MODEL (DATA SCIENTIST ROLE)**

In [ ]:
print(data['high_risk'].sum())

300


In [ ]:
X = data[['age', 'tumor_marker_level', 'family_history', 'smoking_index', 'lifestyle_score', 'sex']]
y = data['high_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    )

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('Classification report for OncoGuide AI high-risk prediction model:')
print(classification_report(y_test, y_pred))
print('NOTE: This is a toy model for education only, not a clinical tool.')

Classification report for OncoGuide AI high-risk prediction model:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98        60
           1       0.98      0.97      0.97        60

    accuracy                           0.97       120
   macro avg       0.98      0.97      0.97       120
weighted avg       0.98      0.97      0.97       120

NOTE: This is a toy model for education only, not a clinical tool.


**FAIRNESS CHECK BY SEX (DATA SCIENTIST & ETHICIST)**

In [ ]:
test_data = X_test.copy()
test_data['true_high_risk'] = y_test.values
test_data['pred_high_risk'] = y_pred

group_stats = test_data.groupby('sex')[['true_high_risk', 'pred_high_risk']].mean()
group_stats.rename(index={0: 'Female', 1: 'Male'}, inplace=True)

print("Average true and predicted 'high risk' rates by sex:")
print(group_stats)

print('[AI Ethicist] Observation:')
print('We must ensure OncoGuide AI does not systematically flag one sex more than the other')
print('without clinical justification. Fairness checks like this help identify potential bias.')


Average true and predicted 'high risk' rates by sex:
        true_high_risk  pred_high_risk
sex                                   
Female        0.569444        0.569444
Male          0.395833        0.375000
[AI Ethicist] Observation:
We must ensure OncoGuide AI does not systematically flag one sex more than the other
without clinical justification. Fairness checks like this help identify potential bias.


**EXPLAIN MY RECOMMENDATION FUNCTION(TRANSPARENCY)**

In [ ]:
def explain_my_recommendation(patient_features, model):
  """
  Simple explanation: show features and predicted risk.
  In a real system, more advanced explanability tools would be needed.
  """
  columns = ['age', 'tumor_marker_level', 'family_history', 'smoking_index', 'lifestyle_score', 'sex']
  patient_df = pd.DataFrame([patient_features], columns=columns)
  prob = model.predict_proba(patient_df)[0, 1]
  pred = int(prob > 0.5)

  print('=== OncoGuide AI Recommendation Explanation ===')
  print(f'Input features: {dict(zip(columns, patient_features))}')
  print(f'Predicted probability of being high risk: {prob:.2f}')
  print(f'Final classification: {"HIGH RISK (flag for follow-up)" if pred == 1 else "NOT HIGH RISK"}')
  print('Reminder: This is a hypothetical educational example, not medical advice.')

example_patient = [62, 65, 1, 10, 4, 0] #older, higher marker, family history, smoker, lower lifestyle score, female
explain_my_recommendation(example_patient, model)

=== OncoGuide AI Recommendation Explanation ===
Input features: {'age': 62, 'tumor_marker_level': 65, 'family_history': 1, 'smoking_index': 10, 'lifestyle_score': 4, 'sex': 0}
Predicted probability of being high risk: 1.00
Final classification: HIGH RISK (flag for follow-up)
Reminder: This is a hypothetical educational example, not medical advice.


**PRODUCT MANAGER PERSPECTIVE (DEPLOYMENT & IMPACT)**

In [ ]:
print('=== Product Manager Perspective ===')
print('OncoGuide AI is positioned as a decision-support tool for clinicians, not a diagnostic authority.')
print('Deployment strategy:')
print('- Pilot in a limited number of hospitals with strong ethics oversight.')
print('- Provide training for clinicians on how to interpret AI flags.')
print('- Intergrate with existing eletronic health record systems.')
print('Societal impact considerations:')
print('- Potential to support earlier detection by highlighting high-risk cases.')
print('- Risk of over-reliance on AI or unnecessary anxiety if flags are not well explained.')
print('- Need to ensure equitable access across different healthcare settings.')

=== Product Manager Perspective ===
OncoGuide AI is positioned as a decision-support tool for clinicians, not a diagnostic authority.
Deployment strategy:
- Pilot in a limited number of hospitals with strong ethics oversight.
- Provide training for clinicians on how to interpret AI flags.
- Intergrate with existing eletronic health record systems.
Societal impact considerations:
- Potential to support earlier detection by highlighting high-risk cases.
- Risk of over-reliance on AI or unnecessary anxiety if flags are not well explained.
- Need to ensure equitable access across different healthcare settings.


**CONSUMER ADVOCATE PERSPECTIVE ( PATIENTS & RIGHTS)**

In [ ]:
print('=== Consumer Advocate Perspective ===')
print('Focus: patient rights, informed consent, and protection from harm.')
print('Key concerns:')
print('- Patients must understand that AI is assisting clinicians, not replacing them.')
print('- Data privacy: sensitive health datamust be protection from harm.')
print("- Risk of unnecessary worry if patients see 'high risk' labels without proper context.")
print('Recommendations:')
print('- Clear communication materials explaining OncoGuide AI to patients.')
print('- Strong consent processes for using patient data in AI systems.')
print('- Mechanisms for patients to ask questions and challenge decisions.')

=== Consumer Advocate Perspective ===
Focus: patient rights, informed consent, and protection from harm.
Key concerns:
- Patients must understand that AI is assisting clinicians, not replacing them.
- Data privacy: sensitive health datamust be protection from harm.
- Risk of unnecessary worry if patients see 'high risk' labels without proper context.
Recommendations:
- Clear communication materials explaining OncoGuide AI to patients.
- Strong consent processes for using patient data in AI systems.
- Mechanisms for patients to ask questions and challenge decisions.


**GOVERNMENT REGULATOR PERSPECTIVE (RULES & SAFEGUARDS)**

In [ ]:
print('=== Government Regulator Perspective ===')
print('Proposed regulatory guidelines for OncoGuide AI:')
print('- Require clinical validation studies brfore widespread deployment.')
print('- Mandate bias and performance reporting across demographic groups.')
print('- Enforce strict data protection and security standards.')
print('- Ensure that AI outputs are always reviewed by qualified medical professionals.')
print('- Prohibit using AI alone for life-changing decisions (e.g., treatment denial).')
print('Goal: Protect patients, ensure safety and fairness, and maintain trust in healthcare.')

=== Government Regulator Perspective ===
Proposed regulatory guidelines for OncoGuide AI:
- Require clinical validation studies brfore widespread deployment.
- Mandate bias and performance reporting across demographic groups.
- Enforce strict data protection and security standards.
- Ensure that AI outputs are always reviewed by qualified medical professionals.
- Prohibit using AI alone for life-changing decisions (e.g., treatment denial).
Goal: Protect patients, ensure safety and fairness, and maintain trust in healthcare.


**AI ETHICIST SUMMARY (CROSS-ROLE SYNTHESIS)**

In [ ]:
print('=== AI Ethicist Summary ===')
print('OncoGuide AI could help clinicians identify patients who may benefit from further cancer screening.')
print('Ethical risks include:')
print('- Bias in training data leading to unequal flagging of certain groups.')
print('- Privacy concerns around highly sensitive medical information.')
print('- Over-reliance on AI outputs without adequate human judgment.')
print('Safeguards must include fairness audits, transparency, strong privacy protections,')
print('and clear boundaries that keep clinicians responsible for final decisions.')

=== AI Ethicist Summary ===
OncoGuide AI could help clinicians identify patients who may benefit from further cancer screening.
Ethical risks include:
- Bias in training data leading to unequal flagging of certain groups.
- Privacy concerns around highly sensitive medical information.
- Over-reliance on AI outputs without adequate human judgment.
Safeguards must include fairness audits, transparency, strong privacy protections,
and clear boundaries that keep clinicians responsible for final decisions.


**PANEL DISCUSSION SCENARIO(ROLE-PLAY)**

In [ ]:
print('=== Panel Discussion Scenario ===')
print("Scenario: OncoGuide AI flags a patient as 'high risk', but follow-up tests show no cancer.")

print('[Data Scientist] We should review false positives and refine the model to reuce unnecessary flags.')
print('[AI Ethicist] We must consider the emotional impact on patients and ensure explanations are compassionate and clear.')
print("[Product Manager] We'll adjust the interface to emphasize that 'high risk' means 'needs further evaluation', not a diagnosis.")
print("[Consumer Advocate] Patients should have access to support and information when they recieve such flags.")
print("[Government Regulator] Guidelines should require monitoring of false positive rates and impact over time.")

=== Panel Discussion Scenario ===
Scenario: OncoGuide AI flags a patient as 'high risk', but follow-up tests show no cancer.
[Data Scientist] We should review false positives and refine the model to reuce unnecessary flags.
[AI Ethicist] We must consider the emotional impact on patients and ensure explanations are compassionate and clear.
[Product Manager] We'll adjust the interface to emphasize that 'high risk' means 'needs further evaluation', not a diagnosis.
[Consumer Advocate] Patients should have access to support and information when they recieve such flags.
[Government Regulator] Guidelines should require monitoring of false positive rates and impact over time.


               **KNOWLEDGE CHECK**

1. Why is it important to check OncoGuide AI's performance across different demographic groups (e.g., sex)?
***Because differences in performance or flagging rates across groups may indicate bias, which could lead to unfair or unsafe treatment for certain patients.

2. Why must OncoGuide AI be used as a decision-support tool rather than a stand alone diagnostic system?
***Because medical diagnosis and treatment decisions require clinical expertise, context, and ethical judgment that AI alone cannot provide; clinicians must remain responsible for final deciscions.

3. What is the one key privacy protection that should be applied to patient data used by OncoGuide AI?
***One key protection is strong data security and access control: patient data should be encrypted, stored securely, and only accessible to authorized healthcare professionals, never sold or shared with third parties.